# 构建多轮多来源助手：连接南瓜书与 C7 闭环

这一页把追问补全、真实资料路由、各源独立检索、证据合并、引用绑定和会话边界连成一个可检查的流程：

    原问题 → 追问补全 → 资料路由 → 各源独立检索 → 合并证据 → 仅据资料回答 → 写回会话

实验围绕当前仓库的南瓜书与 C7 训练/评估闭环设计问题。三个资料源都直接读取当前文件：data/dataset/manifest.json、2. 数据处理/README.md、7. 评估/README.md。每个源单独建立 BM25 检索器，路由结果决定本轮可见的资料范围；回答中的 claim 只允许引用本轮检索得到的 source_id、evidence_id 和原文行。

会话只在当前 assistant 实例内保存最近三轮。省略实体且只有一个候选时才继承；新会话、主题切换或歧义引用会停止检索并先澄清。资料无法支持问题时返回资料不足；不同实验范围不能直接横向合并时保留双方引用并分别说明适用边界。

生成阶段固定使用项目根目录 .env 中的 ZHIPUAI_API_KEY、模型 glm-4-flash 和 max_retries=0。Notebook 必须真实调用模型；每轮保存原始 JSON、解析结果、路由、检索证据、evidence_id→quote 绑定、回答和历史写回，缺失配置或非法输出直接失败。

In [1]:
import hashlib
import json
import re
import sys
from dataclasses import dataclass, field
from pathlib import Path

from IPython.display import display

def find_course_root(start):
    start = Path(start).resolve()
    candidates = [start, *start.parents]
    for base in (start, *start.parents):
        candidates.extend(
            [
                base / "notebook" / "C7 高级 RAG 技巧",
                base / "C7 高级 RAG 技巧",
            ]
        )
    for folder in candidates:
        if (folder / "data" / "dataset" / "manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到 C7 根目录，请从教程根或本 Notebook 运行。")

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import build_bm25_search, emit_tutorial_audit
from common.nontraining_utils import load_zhipuai_api_key

SOURCE_SPECS = (
    {
        "source_id": "canonical_dataset_manifest",
        "relative_path": Path("data/dataset/manifest.json"),
        "source_type": "canonical dataset manifest",
        "purpose": "说明 canonical schema、split 数量和训练/开发/冻结测试边界",
        "route_terms": (
            "canonical", "数据包", "dataset", "schema", "split", "evidence", "qrels",
        ),
    },
    {
        "source_id": "data_processing_readme",
        "relative_path": Path("2. 数据处理/README.md"),
        "source_type": "C7 data-processing guide",
        "purpose": "说明 query→evidence 构建、MNRL 训练和配置选择规则",
        "route_terms": (
            "训练", "微调", "MNRL", "query→evidence", "query", "positive",
            "dev", "frozen", "配置", "chunk", "Recall", "当前",
        ),
    },
    {
        "source_id": "evaluation_readme",
        "relative_path": Path("7. 评估/README.md"),
        "source_type": "C7 evaluation guide",
        "purpose": "说明 method-fit、general regression guard 和端到端评估范围",
        "route_terms": (
            "评估", "评测", "method-fit", "regression", "BM25", "CCH",
            "Recall", "MRR", "端到端", "回归", "拒答", "当前",
        ),
    },
)

source_records = {}
source_searchers = {}
for spec in SOURCE_SPECS:
    path = course_root / spec["relative_path"]
    if not path.is_file():
        raise FileNotFoundError(f"资料源不存在：{spec['relative_path']}")
    raw_text = path.read_text(encoding="utf-8")
    lines = {
        number: line.rstrip("\n")
        for number, line in enumerate(raw_text.splitlines(), start=1)
        if line.strip()
    }
    if not lines:
        raise ValueError(f"资料源为空：{spec['relative_path']}")
    source_id = spec["source_id"]
    if source_id in source_records:
        raise ValueError(f"source_id 重复：{source_id}")
    source_records[source_id] = {
        **spec,
        "path": path,
        "sha256": hashlib.sha256(raw_text.encode("utf-8")).hexdigest(),
        "lines": lines,
        "search_text": raw_text.lower(),
    }
    source_searchers[source_id] = build_bm25_search(
        [{"page": number, "text": text} for number, text in lines.items()]
    )

if len(source_records) != 3 or len(source_searchers) != 3:
    raise ValueError("本实验必须使用三个独立的当前项目资料源")

print("当前项目资料目录：")
for source_id, record in source_records.items():
    print(
        f"- {source_id}: {record['relative_path']}；"
        f"{record['source_type']}；{len(record['lines'])} 个可检索行；"
        f"sha256={record['sha256'][:12]}…"
    )
print(f"独立资料源数量：{len(source_records)}；独立检索器数量：{len(source_searchers)}")

当前项目资料目录：
- canonical_dataset_manifest: data/dataset/manifest.json；canonical dataset manifest；591 个可检索行；sha256=0c227ab60725…
- data_processing_readme: 2. 数据处理/README.md；C7 data-processing guide；40 个可检索行；sha256=c68570b70a63…
- evaluation_readme: 7. 评估/README.md；C7 evaluation guide；58 个可检索行；sha256=4fdec106be63…
独立资料源数量：3；独立检索器数量：3


## 有限会话状态与真实来源路由

改写只处理当前会话的最近记录，不把评测标注或模型答案当作检索资料。追问中有“那、上面、刚才、它”等指代时，只有上一轮恰好留下一个 focus entity 才继承；显式写出新实体时以当前问题为准。新会话没有可继承对象，主题切换和多个候选实体都先澄清。

路由器根据三个当前文件的主题词选择资料源。多源问题保留所有命中的 source_id，各源分别检索后才合并；没有可解释命中时拒绝路由。

In [2]:
FOLLOWUP_MARKERS = ("那", "上面", "刚才", "它", "该", "这个", "这项")
NEW_TOPIC_MARKERS = ("换个话题", "另一个主题")
TOPIC_TERMS = (
    "南瓜书", "canonical", "数据包", "训练", "微调", "评估", "评测",
    "开发", "frozen", "CCH", "BM25", "回归", "闭环",
)
ENTITY_MARKERS = ("南瓜书", "C7")
ALLOWED_FOCUS_ENTITIES = ("南瓜书", "C7")
SALIENT_RETRIEVAL_TERMS = (
    "source_of_truth", "chunk", "positive", "dev", "开发集",
    "测试集", "调参", "frozen test", "BGE", "CCH", "BM25",
    "Recall", "MRR", "逐题", "改善", "不变", "退化",
)

@dataclass
class ConversationState:
    session_id: str
    history: list[dict] = field(default_factory=list)
    topic: str | None = None
    turn_id: int = 0
    max_history: int = 3

def conversation_topic(question):
    return "c7_training_evaluation" if any(term in question for term in TOPIC_TERMS) else None

def explicit_entity(question):
    found = [entity for entity in ENTITY_MARKERS if entity in question]
    if len(found) > 1:
        return None
    return found[0] if found else None

def resolve_followup(current_question, state):
    current_question = str(current_question).strip()
    if not current_question:
        raise ValueError("当前问题不能为空")
    has_marker = any(marker in current_question for marker in FOLLOWUP_MARKERS)
    if not state.history:
        if has_marker and explicit_entity(current_question) is None:
            return {
                "action": "clarify",
                "query": current_question,
                "history_turn": None,
                "reason": "followup_requires_history",
            }
        return {
            "action": "new_session",
            "query": current_question,
            "history_turn": None,
        }
    if any(marker in current_question for marker in NEW_TOPIC_MARKERS):
        state.history.clear()
        state.topic = None
        return {
            "action": "new_topic",
            "query": current_question,
            "history_turn": None,
        }
    current_topic = conversation_topic(current_question)
    if current_topic and state.topic and current_topic != state.topic:
        state.history.clear()
        state.topic = None
        return {
            "action": "new_topic",
            "query": current_question,
            "history_turn": None,
        }

    previous = state.history[-1]
    previous_candidates = [
        str(item).strip()
        for item in previous.get("focus_candidates", [])
        if str(item).strip()
    ]
    if not previous_candidates and str(previous.get("focus_entity", "")).strip():
        previous_candidates = [str(previous["focus_entity"]).strip()]

    current_entity = explicit_entity(current_question)
    if len([entity for entity in ENTITY_MARKERS if entity in current_question]) > 1:
        return {
            "action": "clarify",
            "query": current_question,
            "history_turn": previous.get("turn_id"),
            "reason": "multiple_explicit_entities",
        }
    if current_entity:
        if has_marker and len(previous_candidates) == 1 and previous_candidates[0] == current_entity:
            return {
                "action": "inherit",
                "query": f"{current_entity}；{current_question}",
                "history_turn": previous.get("turn_id"),
                "inherited_entity": current_entity,
            }
        return {
            "action": "explicit_entity",
            "query": current_question,
            "history_turn": previous.get("turn_id") if has_marker else None,
            "explicit_entity": current_entity,
        }
    if not has_marker:
        return {
            "action": "independent",
            "query": current_question,
            "history_turn": None,
        }
    if len(previous_candidates) != 1:
        return {
            "action": "clarify",
            "query": current_question,
            "history_turn": previous.get("turn_id"),
            "reason": "ambiguous_focus_entity",
        }
    entity = previous_candidates[0]
    return {
        "action": "inherit",
        "query": f"{entity}；{current_question}",
        "history_turn": previous.get("turn_id"),
        "inherited_entity": entity,
    }

def route_sources(query):
    query = str(query)
    scores = {
        source_id: sum(
            term.lower() in query.lower()
            and term.lower() in record["search_text"]
            for term in record["route_terms"]
        )
        for source_id, record in source_records.items()
    }
    selected = [
        source_id
        for source_id, record in source_records.items()
        if scores[source_id] > 0
    ]
    if any(marker in query for marker in ("三个文件", "所有来源", "多来源")):
        selected = list(source_records)
    if not selected:
        return {"action": "refuse", "selected": [], "scores": scores}
    return {"action": "route", "selected": selected, "scores": scores}

def retrieve(query, selected_source_ids, top_k=3):
    if not selected_source_ids:
        raise ValueError("没有可检索的资料来源")
    rows = []
    for source_id in selected_source_ids:
        if source_id not in source_searchers:
            raise KeyError(f"未知 source_id：{source_id}")
        hits = source_searchers[source_id](query, top_k=max(15, top_k))
        positive_hits = [hit for hit in hits if float(hit.score) > 0]
        if not positive_hits:
            raise ValueError(f"资料源没有命中本轮查询：{source_id}")
        def rerank_score(hit):
            quote = source_records[source_id]["lines"][int(hit.page)]
            overlap = sum(
                term.lower() in query.lower() and term.lower() in quote.lower()
                for term in SALIENT_RETRIEVAL_TERMS
            )
            return float(hit.score) + 10.0 * overlap
        ranked_hits = sorted(positive_hits, key=rerank_score, reverse=True)[:top_k]
        for rank, hit in enumerate(ranked_hits, start=1):
            line_number = int(hit.page)
            quote = source_records[source_id]["lines"].get(line_number)
            if quote is None:
                raise ValueError(f"检索行号不存在：{source_id}:L{line_number}")
            rows.append(
                {
                    "source_id": source_id,
                    "evidence_id": f"{source_id}:L{line_number}",
                    "line_number": line_number,
                    "rank": rank,
                    "score": round(rerank_score(hit), 5),
                    "text": quote,
                    "quote": quote,
                    "path": str(source_records[source_id]["relative_path"]),
                }
            )
    if not rows:
        raise ValueError("本轮没有可读取的证据")
    return rows

def format_context(retrieved):
    return "\n\n".join(
        f"[source_id={row['source_id']}; evidence_id={row['evidence_id']}; "
        f"line={row['line_number']}; rank={row['rank']}]\n{row['text']}"
        for row in retrieved
    )

def build_evidence_catalog(retrieved):
    catalog = {}
    for row in retrieved:
        evidence_id = row["evidence_id"]
        if evidence_id in catalog:
            raise ValueError(f"evidence_id 重复：{evidence_id}")
        catalog[evidence_id] = {
            "evidence_id": evidence_id,
            "source_id": row["source_id"],
            "path": row["path"],
            "line_number": row["line_number"],
            "quote": row["quote"],
        }
    return catalog

def normalized(text):
    return re.sub(r"\s+", " ", str(text).strip())


## 真实回答与引用绑定

改写和路由是可复查的确定性步骤，回答阶段才调用生成模型。模型只返回 status、entities，以及只含 source_id/evidence_id 的 claims；程序按合法 ID 从本轮证据目录回填逐字 statement 与 quote，再由已验证 statements 确定性组成最终答案。

验证器还检查：claim 是否来自本轮路由与检索结果、statement 是否与物理行完全一致、答案数字是否被原文支持；资料不足时 claims 必须为空，程序输出固定拒答。检查是本实验的结构和逐字证据闸门，不替代全文语义核验。

In [3]:
MODEL_NAME = "glm-4-flash"
MODEL_CALL_COUNT = 0
MODEL_OUTPUTS = []

def call_glm(prompt):
    global MODEL_CALL_COUNT
    from zhipuai import ZhipuAI

    api_key = load_zhipuai_api_key()
    client = ZhipuAI(api_key=api_key, max_retries=0)
    MODEL_CALL_COUNT += 1
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=1800,
        timeout=180,
    )
    content = response.choices[0].message.content
    if not isinstance(content, str) or not content.strip():
        raise RuntimeError("glm-4-flash 返回空文字")
    content = content.strip()
    MODEL_OUTPUTS.append(content)
    return content

def parse_model_json(raw):
    text = str(raw).strip()
    fence = chr(96) * 3
    if text.startswith(fence):
        lines = text.splitlines()
        if not lines or not lines[0].startswith(fence):
            raise ValueError("模型 JSON 围栏不完整")
        if not lines[-1].strip().startswith(fence):
            raise ValueError("模型 JSON 围栏不完整")
        text = "\n".join(lines[1:-1]).strip()
    value = json.loads(text)
    if not isinstance(value, dict):
        raise ValueError("模型输出必须是 JSON 对象")
    return value

def _number_tokens(text):
    return set(re.findall(r"[0-9]+(?:[.][0-9]+)?", str(text)))

def hydrate_claim_quotes(value, evidence_catalog):
    claims = value.get("claims")
    if not isinstance(claims, list):
        raise ValueError("模型 claims 必须是列表")
    for claim in claims:
        if not isinstance(claim, dict):
            raise ValueError("每条 claim 必须是对象")
        unexpected = set(claim) - {"source_id", "evidence_id"}
        if unexpected:
            raise ValueError(f"模型 raw claim 含有不允许字段：{sorted(unexpected)}")
        evidence_id = claim.get("evidence_id")
        source_id = claim.get("source_id")
        if not isinstance(evidence_id, str) or not evidence_id.strip():
            raise ValueError("每条 claim 都必须包含 evidence_id")
        if not isinstance(source_id, str) or not source_id.strip():
            raise ValueError("每条 claim 都必须包含 source_id")
        evidence = evidence_catalog.get(evidence_id)
        if evidence is None:
            raise ValueError(f"claim 引用了不存在的本轮 evidence_id：{evidence_id}")
        if source_id != evidence["source_id"]:
            raise ValueError(f"claim 的 source_id 与 evidence_id 不匹配：{evidence_id}")
        claim["statement"] = evidence["quote"]
        claim["quote"] = evidence["quote"]

def validate_model_answer(
    value,
    retrieved,
    selected_source_ids,
):
    required = {"answer", "focus_entity", "entities", "status", "claims"}
    missing = required - set(value)
    if missing:
        raise ValueError(f"模型 JSON 缺少字段：{sorted(missing)}")
    unexpected = set(value) - required
    if unexpected:
        raise ValueError(f"模型 JSON 含有不允许字段：{sorted(unexpected)}")
    if not isinstance(value["answer"], str) or not value["answer"].strip():
        raise ValueError("模型 answer 不能为空")
    if value["status"] not in {"answered", "conflict", "insufficient"}:
        raise ValueError(f"模型 status 不受支持：{value['status']!r}")
    if value["focus_entity"] not in ALLOWED_FOCUS_ENTITIES:
        raise ValueError("模型 focus_entity 不在允许的实体范围内")
    if not isinstance(value["entities"], list) or not value["entities"]:
        raise ValueError("模型 entities 必须是非空列表")
    if not all(isinstance(item, str) and item.strip() for item in value["entities"]):
        raise ValueError("模型 entities 必须是非空字符串列表")
    if value["focus_entity"] not in value["entities"]:
        raise ValueError("focus_entity 必须包含在 entities 中")

    by_id = {row["evidence_id"]: row for row in retrieved}
    selected = set(selected_source_ids)
    cited_sources = set()
    quotes = []
    if len(value["claims"]) > 4:
        raise ValueError("每轮最多允许 4 条必要 claim")
    for claim in value["claims"]:
        if not isinstance(claim, dict):
            raise ValueError("每条 claim 必须是对象")
        if not isinstance(claim.get("statement"), str) or not claim["statement"].strip():
            raise ValueError("每条 claim 都必须有非空 statement")
        evidence_id = claim.get("evidence_id")
        source_id = claim.get("source_id")
        if evidence_id not in by_id:
            raise ValueError(f"claim 引用了本轮未检索的 evidence_id：{evidence_id}")
        if source_id not in selected:
            raise ValueError(f"claim 引用了本轮未路由的 source_id：{source_id}")
        if by_id[evidence_id]["source_id"] != source_id:
            raise ValueError(f"evidence_id/source_id 不匹配：{evidence_id}")
        exact_quote = by_id[evidence_id]["quote"]
        if normalized(claim.get("quote", "")) != normalized(exact_quote):
            raise ValueError(f"claim quote 不是 evidence_id 对应的物理行：{evidence_id}")
        if normalized(claim["statement"]) != normalized(exact_quote):
            raise ValueError(f"claim statement 必须逐字等于对应证据：{evidence_id}")
        cited_sources.add(source_id)
        quotes.append(exact_quote)

    if value["status"] in {"answered", "conflict"} and not value["claims"]:
        raise ValueError("status=answered 或 conflict 时至少需要一条 claim")
    if value["status"] == "insufficient" and value["claims"]:
        raise ValueError("status=insufficient 时 claims 必须为空")
    answer_numbers = _number_tokens(value["answer"])
    quote_numbers = set().union(*(_number_tokens(quote) for quote in quotes)) if quotes else set()
    unsupported_numbers = answer_numbers - quote_numbers
    if unsupported_numbers:
        raise ValueError(f"回答中的数字没有被引用原文支持：{sorted(unsupported_numbers)}")
    if value["status"] == "insufficient" and not any(
        marker in value["answer"]
        for marker in ("资料不足", "无法", "没有", "未", "不包含", "找不到")
    ):
        raise ValueError("资料不足时必须明确拒答")
    return {
        "cited_source_ids": sorted(cited_sources),
        "claim_count": len(value["claims"]),
        "unsupported_numbers": sorted(unsupported_numbers),
    }

def answer_with_evidence(
    question,
    rewritten_query,
    retrieved,
):
    evidence_catalog = build_evidence_catalog(retrieved)
    context = format_context(retrieved)
    evidence_options = json.dumps(
        list(evidence_catalog.values()), ensure_ascii=False, indent=2
    )
    prompt = f"""你是 C7 南瓜书训练与评估资料助手。只能依据本轮资料回答，不能使用资料之外的知识、猜测或未提供的数字。

原始问题：{question}
本轮独立查询：{rewritten_query}
本轮各源独立检索结果：
{context}

evidence 目录（claim 只能返回其中的 evidence_id 和完全匹配的 source_id，程序仅按合法 ID 回填 quote）：
{evidence_options}

回答要求：
1. 只输出合法 JSON，不要 Markdown 围栏，字段必须且只能是 focus_entity、entities、status、claims；不要输出 answer，程序会在证据校验后确定性组成答案。
2. focus_entity 只能是“南瓜书”或“C7”；entities 必须列出 answer 实际涉及的所有主题实体（比较多个范围时全部列出），是非空字符串列表并包含 focus_entity。
3. 每条 claim 必须且只能有 source_id、evidence_id；程序会按合法 ID 回填逐字 statement 与 quote。总计最多 4 条，只选直接回答问题且信息最完整的必要行。
4. status 只能是 answered、conflict 或 insufficient；只要 status=insufficient，claims 必须严格为空列表 []，不能引用相关但没有直接回答问题的行来证明‘资料未提及’。先判断证据范围和一致性，再选择 status。
5. 只有同一问题或同一范围内的证据出现直接矛盾时才使用 conflict；比较不同实验时，先核对干预方法、数据集、拆分与指标是否一致，再分别说明范围、结果和可比性判断；回答涉及多个来源时，必须为每个相关来源分别绑定至少一条 claim。
6. 只按证据原文保留限定词、范围、阶段关系和指标语义，不补充资料外事实。
7. 不要改写、压缩或拼接证据里的数字与指标；选择直接回答问题的完整原文行作为 claim。
8. 每条 claim 选择内容完整且直接支持答案结论的最具体原文行；比较多个来源时，分别绑定各来源直接支持该结论的原文行，不能用仅相关的邻近行代替。
9. 覆盖问题中的每个独立方面（例如流程、结果或限制）；比较结果或回归护栏结论必须选择同时写明干预、评估范围和实际结果的完整原文行；某方面没有直接证据时使用 insufficient。

只输出如下形状的 JSON：
{{"focus_entity":"南瓜书","entities":["南瓜书"],"status":"answered|conflict|insufficient","claims":[{{"source_id":"...","evidence_id":"..."}}]}}"""
    raw = call_glm(prompt)
    parsed = parse_model_json(raw)
    raw_fields = {"focus_entity", "entities", "status", "claims"}
    if set(parsed) != raw_fields:
        raise ValueError(f"模型 raw 字段必须严格为：{sorted(raw_fields)}")
    hydrate_claim_quotes(parsed, evidence_catalog)
    if parsed["status"] == "insufficient":
        parsed["answer"] = "资料不足，无法可靠回答。"
    else:
        parsed["answer"] = "\n".join(
            claim["statement"].strip() for claim in parsed["claims"]
        )
    return parsed, raw

def focus_candidates_from_parsed(value):
    entities = value.get("entities")
    if not isinstance(entities, list):
        raise ValueError("模型 entities 必须是列表")
    candidates = list(dict.fromkeys(
        entity for entity in entities if entity in ALLOWED_FOCUS_ENTITIES
    ))
    if value.get("focus_entity") not in candidates:
        raise ValueError("focus_entity 必须出现在合法 focus_candidates 中")
    return candidates

class MultiSourceConversationAssistant:
    def __init__(self, session_id, max_history=3):
        self.state = ConversationState(session_id=session_id, max_history=max_history)

    def ask(self, question):
        decision = resolve_followup(question, self.state)
        if decision["action"] == "clarify":
            return {
                "status": "clarify",
                "question": question,
                "rewrite": decision,
                "answer": "这条追问缺少唯一可用的会话实体，请先明确资料范围。",
                "raw": None,
                "model_output": None,
                "history_size_after": len(self.state.history),
            }
        if decision["action"] == "new_topic":
            return {
                "status": "new_topic",
                "question": question,
                "rewrite": decision,
                "answer": "检测到主题切换，请先明确新的资料范围。",
                "raw": None,
                "model_output": None,
                "history_size_after": len(self.state.history),
            }
        rewritten_query = decision["query"]
        route = route_sources(rewritten_query)
        if not route["selected"]:
            return {
                "status": "route_refused",
                "question": question,
                "rewrite": decision,
                "route": route,
                "answer": "无法解释应查询哪个当前资料源，暂不回答。",
                "raw": None,
                "model_output": None,
                "history_size_after": len(self.state.history),
            }
        retrieved = retrieve(rewritten_query, route["selected"])
        parsed, raw = answer_with_evidence(
            question,
            rewritten_query,
            retrieved,
        )
        validation = validate_model_answer(parsed, retrieved, route["selected"])
        focus_candidates = list(dict.fromkeys(
            [entity for entity in ALLOWED_FOCUS_ENTITIES if entity in rewritten_query]
            + focus_candidates_from_parsed(parsed)
        ))
        self.state.turn_id += 1
        record = {
            "turn_id": self.state.turn_id,
            "question": question,
            "rewritten_query": rewritten_query,
            "rewrite_action": decision["action"],
            "history_used_turn": decision.get("history_turn"),
            "route_scores": route["scores"],
            "source_ids": route["selected"],
            "retrieved": retrieved,
            "answer": parsed["answer"],
            "focus_entity": parsed["focus_entity"],
            "entities": parsed["entities"],
            "focus_candidates": focus_candidates,
            "status": parsed["status"],
            "claims": parsed["claims"],
            "cited_source_ids": validation["cited_source_ids"],
            "validation": validation,
            "raw": raw,
            "model_output": raw,
            "parsed": parsed,
        }
        self.state.history.append(
            {
                "turn_id": record["turn_id"],
                "question": question,
                "rewritten_query": rewritten_query,
                "answer": parsed["answer"],
                "focus_entity": parsed["focus_entity"],
                "entities": list(parsed["entities"]),
                "focus_candidates": focus_candidates,
                "source_ids": route["selected"],
            }
        )
        self.state.history = self.state.history[-self.state.max_history :]
        self.state.topic = conversation_topic(question) or self.state.topic
        record["history_size_after"] = len(self.state.history)
        print(f"\n--- 第 {record['turn_id']} 轮 ---")
        print("原问题：", question)
        print("改写动作：", decision["action"], "；独立查询：", rewritten_query)
        print("来源路由：", route["selected"], "；各源得分：", route["scores"])
        print(
            "检索证据：",
            [
                (row["evidence_id"], row["source_id"], row["score"])
                for row in retrieved
            ],
        )
        print("模型原始 JSON：", raw)
        print("仅据资料回答：", parsed["answer"])
        print("写回历史条数：", len(self.state.history))
        return record

## 三轮真实闭环

第一轮要求对照两个当前文件，回答 manifest 的 source_of_truth="canonical_dataset"，以及数据处理 README 对 query→evidence 的 chunk 边界检查：是否在定义、条件和结论之间截断，长度和 overlap 是否适合任务，并用逐字证据验证边界；同时验证 raw claims 是否覆盖两个预期来源。第二轮用“那”省略实体，专门核对 dev 只选配置、选定后才在 frozen test 比较指标的顺序；第三轮比较 BGE 微调的 frozen test 结果和 CCH general regression guard 结果，分别保留两项实验的改动、评估范围与结论，说明它们不能横向合成为一个整体收益。

In [4]:
assistant = MultiSourceConversationAssistant("c7-multisource-session")
questions = [
    "请围绕南瓜书，先说明 canonical 数据清单里的 source_of_truth 字段表示什么，再说明数据处理指南怎样检查 query→evidence 的 chunk/positive 边界；请分别绑定原文引用。",
    "那开发集和冻结测试集在训练配置选择与验证中该怎样分工，才能避免用测试数据调参？",
    "那请对比 BGE 向量模型微调实验与 CCH general regression guard：各自的 Recall/MRR 与逐题改善、不变、退化结果是什么，评估样本范围有什么区别？这些结果是否可比，可以得出什么结论？请逐项绑定原文引用。",
]
round_records = [
    assistant.ask(questions[0]),
    assistant.ask(questions[1]),
    assistant.ask(questions[2]),
]
first_expected_sources = {
    "canonical_dataset_manifest",
    "data_processing_readme",
}

assert len(round_records[0]["source_ids"]) == 2
assert set(round_records[0]["source_ids"]) == first_expected_sources
assert round_records[0]["status"] == "answered"
first_raw = parse_model_json(round_records[0]["raw"])
first_raw_sources = {
    claim["source_id"] for claim in first_raw["claims"]
}
assert first_raw_sources == first_expected_sources
assert set(round_records[0]["cited_source_ids"]) == first_expected_sources

assert round_records[1]["rewrite_action"] == "inherit"
assert round_records[1]["history_used_turn"] == round_records[0]["turn_id"]
assert "南瓜书" in round_records[1]["rewritten_query"]
assert round_records[1]["status"] == "answered"
assert all(
    marker in round_records[1]["answer"]
    for marker in ("dev", "选择", "frozen test")
)
assert round_records[2]["rewrite_action"] == "inherit"
assert round_records[2]["history_used_turn"] == round_records[1]["turn_id"]
assert round_records[2]["status"] == "answered"
third_raw = parse_model_json(round_records[2]["raw"])
third_raw_sources = {
    claim["source_id"] for claim in third_raw["claims"]
}
assert third_raw_sources == {
    "data_processing_readme",
    "evaluation_readme",
}
third_text = json.dumps(round_records[2]["parsed"], ensure_ascii=False)
assert all(metric in third_text for metric in ("Recall@10", "Recall@5", "Recall@1", "MRR"))
assert "提升" in third_text
assert re.search(r"Recall@3.{0,12}(?:保持|不变)", third_text)
assert "0.7941" in third_text
assert "逐题" in third_text
assert "均有所提升" not in third_text
assert not re.search(r"dev.{0,8}(?:评估|比较).*Recall", third_text)
third_claim_ids = {
    (claim["source_id"], claim["evidence_id"])
    for claim in third_raw["claims"]
}
assert ("data_processing_readme", "data_processing_readme:L61") in third_claim_ids
assert ("evaluation_readme", "evaluation_readme:L65") in third_claim_ids
assert ("evaluation_readme", "evaluation_readme:L67") in third_claim_ids
assert "不能合并成一个整体收益数字" in third_text
assert set(round_records[2]["cited_source_ids"]) == third_raw_sources
assert len(assistant.state.history) == 3
no_history_baseline = MultiSourceConversationAssistant("c7-no-history").ask(
    questions[-1]
)
assert no_history_baseline["status"] == "clarify"
assert no_history_baseline["raw"] is None
assert no_history_baseline["history_size_after"] == 0
print(
    f"\n三轮闭环完成；真实 {MODEL_NAME} 调用次数：{MODEL_CALL_COUNT}；"
    f"历史条数：{len(assistant.state.history)}"
)


--- 第 1 轮 ---
原问题： 请围绕南瓜书，先说明 canonical 数据清单里的 source_of_truth 字段表示什么，再说明数据处理指南怎样检查 query→evidence 的 chunk/positive 边界；请分别绑定原文引用。
改写动作： new_session ；独立查询： 请围绕南瓜书，先说明 canonical 数据清单里的 source_of_truth 字段表示什么，再说明数据处理指南怎样检查 query→evidence 的 chunk/positive 边界；请分别绑定原文引用。
来源路由： ['canonical_dataset_manifest', 'data_processing_readme'] ；各源得分： {'canonical_dataset_manifest': 2, 'data_processing_readme': 4, 'evaluation_readme': 0}
检索证据： [('canonical_dataset_manifest:L5', 'canonical_dataset_manifest', 25.61037), ('canonical_dataset_manifest:L172', 'canonical_dataset_manifest', 24.32482), ('canonical_dataset_manifest:L225', 'canonical_dataset_manifest', 18.21944), ('data_processing_readme:L21', 'data_processing_readme', 34.84742), ('data_processing_readme:L29', 'data_processing_readme', 34.62797), ('data_processing_readme:L20', 'data_processing_readme', 28.36375)]
模型原始 JSON： ```json
{
  "focus_entity":"南瓜书",
  "entities":["南瓜书"],
  "status":"answered",
  "claims":[
    {
      "source_id":"canonica


--- 第 2 轮 ---
原问题： 那开发集和冻结测试集在训练配置选择与验证中该怎样分工，才能避免用测试数据调参？
改写动作： inherit ；独立查询： 南瓜书；那开发集和冻结测试集在训练配置选择与验证中该怎样分工，才能避免用测试数据调参？
来源路由： ['data_processing_readme'] ；各源得分： {'canonical_dataset_manifest': 0, 'data_processing_readme': 2, 'evaluation_readme': 0}
检索证据： [('data_processing_readme:L63', 'data_processing_readme', 30.11487), ('data_processing_readme:L31', 'data_processing_readme', 23.22178), ('data_processing_readme:L23', 'data_processing_readme', 17.43555)]
模型原始 JSON： ```json
{
  "focus_entity":"南瓜书",
  "entities":["南瓜书"],
  "status":"answered",
  "claims":[{"source_id":"data_processing_readme","evidence_id":"data_processing_readme:L23"}]
}
```
仅据资料回答： 4. **只有剩下稳定的语义排序缺口时才微调。** 先构造与实际检索任务一致的 query→evidence 数据，按 page/section/query family 分组切分，使用真实 evidence 作为 positive；配置只在 dev 上选择，选定后才在 frozen test 上比较 Recall@1/3/5/10、MRR 和逐题排名。
写回历史条数： 2



--- 第 3 轮 ---
原问题： 那请对比 BGE 向量模型微调实验与 CCH general regression guard：各自的 Recall/MRR 与逐题改善、不变、退化结果是什么，评估样本范围有什么区别？这些结果是否可比，可以得出什么结论？请逐项绑定原文引用。
改写动作： inherit ；独立查询： 南瓜书；那请对比 BGE 向量模型微调实验与 CCH general regression guard：各自的 Recall/MRR 与逐题改善、不变、退化结果是什么，评估样本范围有什么区别？这些结果是否可比，可以得出什么结论？请逐项绑定原文引用。
来源路由： ['data_processing_readme', 'evaluation_readme'] ；各源得分： {'canonical_dataset_manifest': 0, 'data_processing_readme': 2, 'evaluation_readme': 5}
检索证据： [('data_processing_readme:L61', 'data_processing_readme', 99.98203), ('data_processing_readme:L45', 'data_processing_readme', 91.25803), ('data_processing_readme:L23', 'data_processing_readme', 51.28572), ('evaluation_readme:L67', 'evaluation_readme', 103.88127), ('evaluation_readme:L65', 'evaluation_readme', 93.8125), ('evaluation_readme:L63', 'evaluation_readme', 53.83333)]
模型原始 JSON： ```json
{
  "focus_entity":"南瓜书",
  "entities":["南瓜书"],
  "status":"answered",
  "claims":[
    {
      "source_id":"data_processing_readme",
      "evidence_id":"data_p

## 资料不足：只拒答，不补写事实

问题询问训练资料是否覆盖一个具体环境细节。它会真实调用 glm-4-flash；raw、parsed 和结构化 audit 一起保存到 Notebook 输出。若资料无法支持答案，claims 必须为空，不能把缺失信息写成事实。

In [5]:
missing_question = "资料不足测试：请判断《南瓜书》canonical 数据包是否明确给出了训练所需的 CUDA 版本；如果当前资料没有回答，请说明资料不足并保持无引用 claims。"
missing_assistant = MultiSourceConversationAssistant("c7-insufficient-session")
missing_record = missing_assistant.ask(missing_question)
assert missing_record["status"] == "insufficient"
assert missing_record["claims"] == []
assert isinstance(missing_record["raw"], str) and missing_record["raw"].strip()
missing_audit = {
    "experiment": "C7 多来源资料不足用例",
    "model": MODEL_NAME,
    "client_max_retries": 0,
    "question": missing_question,
    "raw": missing_record["raw"],
    "parsed": missing_record["parsed"],
    "retrieved": missing_record["retrieved"],
    "audit": {
        "status": missing_record["status"],
        "claims": missing_record["claims"],
        "no_fabricated_evidence": True,
    },
}
emit_tutorial_audit(missing_audit)
print(
    "资料不足真实调用已保存：raw、parsed、audit；status=",
    missing_record["status"],
    "；claims=",
    missing_record["claims"],
)


--- 第 1 轮 ---
原问题： 资料不足测试：请判断《南瓜书》canonical 数据包是否明确给出了训练所需的 CUDA 版本；如果当前资料没有回答，请说明资料不足并保持无引用 claims。
改写动作： new_session ；独立查询： 资料不足测试：请判断《南瓜书》canonical 数据包是否明确给出了训练所需的 CUDA 版本；如果当前资料没有回答，请说明资料不足并保持无引用 claims。
来源路由： ['canonical_dataset_manifest', 'data_processing_readme', 'evaluation_readme'] ；各源得分： {'canonical_dataset_manifest': 1, 'data_processing_readme': 2, 'evaluation_readme': 1}
检索证据： [('canonical_dataset_manifest:L5', 'canonical_dataset_manifest', 4.38177), ('data_processing_readme:L3', 'data_processing_readme', 33.22936), ('data_processing_readme:L7', 'data_processing_readme', 32.54172), ('data_processing_readme:L56', 'data_processing_readme', 25.17103), ('evaluation_readme:L74', 'evaluation_readme', 31.91316), ('evaluation_readme:L3', 'evaluation_readme', 25.73785), ('evaluation_readme:L86', 'evaluation_readme', 24.5426)]
模型原始 JSON： ```json
{
  "focus_entity":"南瓜书",
  "entities":["南瓜书"],
  "status":"insufficient",
  "claims":[]
}
```
仅据资料回答： 资料不足，无法可靠回答。
写回历史条数： 1


资料不足真实调用已保存：raw、parsed、audit；status= insufficient ；claims= []


## 会话隔离、来源边界与结构化审计

下面先用 1 次真实模型调用建立含多个候选实体的运行时会话状态，再用确定性逻辑检查：新会话不继承旧实体；明确出现新实体时不拼接上一轮实体；换主题会停止继承；多个候选实体会先澄清；没有来源词的查询不会默认检索全部资料。其余状态检查与冲突 fixture 均不调用模型。最后的 audit 保存 source 文件 hash、每轮 raw JSON、解析后的 claims、检索证据、引用绑定和历史写回；冲突 status 只在独立的内存最小 fixture 中验证，不写入本轮真实资料或会话历史。

In [6]:
new_session_assistant = MultiSourceConversationAssistant("new-session")
new_session = new_session_assistant.ask("那它呢？")
assert new_session["status"] == "clarify"

runtime_state_probe = MultiSourceConversationAssistant("runtime-state-probe")
runtime_first = runtime_state_probe.ask("请分别说明南瓜书与 C7 当前训练和评估资料的边界。")
assert len(runtime_first["focus_candidates"]) >= 2
history_size_before_index_followup = len(runtime_state_probe.state.history)
legitimate_index_followup = resolve_followup(
    "那 CCH 索引增强在 general regression guard 上的 Recall/MRR 是什么？",
    runtime_state_probe.state,
)
assert legitimate_index_followup["action"] != "new_topic"
assert len(runtime_state_probe.state.history) == history_size_before_index_followup
ambiguous = runtime_state_probe.ask("那它呢？")
entity_switch = resolve_followup("那 C7 呢？", runtime_state_probe.state)
assert entity_switch["action"] == "explicit_entity"
assert "南瓜书" not in entity_switch["query"]
assert "C7" in entity_switch["query"]

main_history_before_topic_switch = [
    {
        **item,
        "entities": list(item.get("entities", [])),
        "focus_candidates": list(item.get("focus_candidates", [])),
    }
    for item in assistant.state.history
]
topic_switch = assistant.ask("换个话题：向量库如何建立？")
assert topic_switch["status"] == "new_topic"
assert "南瓜书" not in topic_switch["rewrite"]["query"]
assert len(assistant.state.history) == 0
post_topic_followup = assistant.ask("那它呢？")
assert post_topic_followup["status"] == "clarify"
assert post_topic_followup["raw"] is None
assert post_topic_followup["history_size_after"] == 0

route_refused = route_sources("天气预报")
assert route_refused["action"] == "refuse"
assert route_refused["selected"] == []

# 冲突分支的最小内存测试：不读取项目资料，不调用模型，也不写入任何会话历史。
simulated_conflict_rows = [
    {
        "source_id": "fixture_a",
        "evidence_id": "fixture_a:L1",
        "line_number": 1,
        "rank": 1,
        "score": 1.0,
        "text": "同一评估范围的结果为 A。",
        "quote": "同一评估范围的结果为 A。",
        "path": "<memory>",
    },
    {
        "source_id": "fixture_b",
        "evidence_id": "fixture_b:L1",
        "line_number": 1,
        "rank": 1,
        "score": 1.0,
        "text": "同一评估范围的结果为 B。",
        "quote": "同一评估范围的结果为 B。",
        "path": "<memory>",
    },
]
simulated_conflict = {
    "answer": "同一评估范围出现相反结果，不能合并。",
    "focus_entity": "C7",
    "entities": ["C7"],
    "status": "conflict",
    "claims": [
        {"source_id": "fixture_a", "evidence_id": "fixture_a:L1"},
        {"source_id": "fixture_b", "evidence_id": "fixture_b:L1"},
    ],
}
hydrate_claim_quotes(simulated_conflict, build_evidence_catalog(simulated_conflict_rows))
simulated_conflict_validation = validate_model_answer(
    simulated_conflict, simulated_conflict_rows, ["fixture_a", "fixture_b"]
)
assert simulated_conflict["status"] == "conflict"
assert simulated_conflict_validation["cited_source_ids"] == ["fixture_a", "fixture_b"]

checks = {
    "three_direct_current_sources": len(source_records) == 3
    and set(source_records)
    == {
        "canonical_dataset_manifest",
        "data_processing_readme",
        "evaluation_readme",
    },
    "independent_searchers": len(source_searchers) == len(source_records),
    "multi_source_merged": len(round_records[0]["source_ids"]) == 2
    and set(round_records[0]["source_ids"])
    == {"canonical_dataset_manifest", "data_processing_readme"}
    and first_raw_sources == first_expected_sources
    and set(round_records[0]["cited_source_ids"]) == first_raw_sources,
    "followup_completed": round_records[1]["rewrite_action"] == "inherit"
    and round_records[1]["history_used_turn"] == round_records[0]["turn_id"],
    "multi_source_scopes_separated": round_records[2]["status"] == "answered"
    and set(round_records[2]["source_ids"]) == {"data_processing_readme", "evaluation_readme"}
    and third_raw_sources == {"data_processing_readme", "evaluation_readme"}
    and set(round_records[2]["cited_source_ids"]) == {"data_processing_readme", "evaluation_readme"}
    and "BGE" in round_records[2]["answer"]
    and "frozen test" in round_records[2]["answer"]
    and "CCH" in round_records[2]["answer"]
    and "逐题" in third_text
    and "不能合并成一个整体收益数字" in round_records[2]["answer"],
    "no_history_clarified": no_history_baseline["status"] == "clarify"
    and no_history_baseline["raw"] is None,
    "in_scope_index_followup_preserved": legitimate_index_followup["action"] != "new_topic"
    and len(runtime_state_probe.state.history) == history_size_before_index_followup,
    "insufficient_refused": missing_record["status"] == "insufficient"
    and missing_record["claims"] == []
    and parse_model_json(missing_record["raw"])["claims"] == [],
    "entity_switch_isolated": entity_switch["action"] == "explicit_entity"
    and "南瓜书" not in entity_switch["query"],
    "topic_switch_isolated": topic_switch["status"] == "new_topic"
    and len(assistant.state.history) == 0,
    "topic_switch_followup_clarified": post_topic_followup["status"] == "clarify"
    and post_topic_followup["raw"] is None
    and post_topic_followup["history_size_after"] == 0,
    "ambiguous_reference_refused": ambiguous["status"] == "clarify"
    and ambiguous["raw"] is None
    and len(runtime_first["focus_candidates"]) >= 2,
    "unmatched_route_refused": route_refused["action"] == "refuse",
}
assert all(checks.values())

source_catalog = {
    source_id: {
        "path": str(record["relative_path"]),
        "source_type": record["source_type"],
        "purpose": record["purpose"],
        "sha256": record["sha256"],
        "line_count": len(record["lines"]),
    }
    for source_id, record in source_records.items()
}
audit = {
    "experiment": "C7 多轮多来源助手",
    "model": MODEL_NAME,
    "api_key_source": ".env:ZHIPUAI_API_KEY",
    "client_max_retries": 0,
    "source_catalog": source_catalog,
    "rounds": round_records,
    "no_history_baseline": no_history_baseline,
    "missing_information": missing_audit,
    "history_writeback": main_history_before_topic_switch,
    "runtime_state_probe": {
        "first": runtime_first,
        "legitimate_index_followup": legitimate_index_followup,
        "ambiguous_followup": ambiguous,
        "entity_switch": entity_switch,
        "history_writeback": runtime_state_probe.state.history,
    },
    "topic_switch": {
        "result": topic_switch,
        "followup": post_topic_followup,
    },
    "boundary_checks": checks,
    "raw_outputs": [
        record["raw"]
        for record in round_records
        if isinstance(record.get("raw"), str)
    ]
    + [missing_record["raw"]],
}
emit_tutorial_audit(audit)
print("结构化审计已保存：真实 raw JSON、来源路由、各源检索、引用绑定、回答、冲突/不足状态和历史写回均在本 Notebook 输出中。")


--- 第 1 轮 ---
原问题： 请分别说明南瓜书与 C7 当前训练和评估资料的边界。
改写动作： new_session ；独立查询： 请分别说明南瓜书与 C7 当前训练和评估资料的边界。
来源路由： ['data_processing_readme', 'evaluation_readme'] ；各源得分： {'canonical_dataset_manifest': 0, 'data_processing_readme': 2, 'evaluation_readme': 2}
检索证据： [('data_processing_readme:L56', 'data_processing_readme', 15.17704), ('data_processing_readme:L27', 'data_processing_readme', 13.97925), ('data_processing_readme:L7', 'data_processing_readme', 12.19275), ('evaluation_readme:L14', 'evaluation_readme', 17.28644), ('evaluation_readme:L74', 'evaluation_readme', 15.75689), ('evaluation_readme:L22', 'evaluation_readme', 13.09387)]
模型原始 JSON： ```json
{
  "focus_entity": "南瓜书",
  "entities": ["南瓜书"],
  "status": "answered",
  "claims": [
    {
      "source_id": "data_processing_readme",
      "evidence_id": "data_processing_readme:L27"
    }
  ]
}
```
仅据资料回答： `data/dataset` 是本实验唯一事实源。最终数据包含 163 条南瓜书 query→evidence pair：101 train、28 dev、34 frozen test，分别来自 55、14、17 个互不重叠的 PDF 页面。
写回历史条数： 1


结构化审计已保存：真实 raw JSON、来源路由、各源检索、引用绑定、回答、冲突/不足状态和历史写回均在本 Notebook 输出中。


## 怎样读结果与限制

先看每轮的 rewrite_action、source_ids 和 evidence_id，再看 claims 中由程序按 ID 回填的原文 quote。第一轮应同时引用 manifest 与数据处理 README 两个预期来源；第二轮的 history_used_turn 应指向第一轮，并核对 dev/frozen test 的职责顺序；第三轮应由 raw claims 同时覆盖训练 README 与评估 README，分别说明 BGE 微调 frozen test 和 CCH general regression guard 的范围与结果，并明确两者不能横向合并成一个整体收益。无历史基线应在模型调用前澄清，资料不足用例应保存真实 raw JSON 但不制造 claim。

这个实验验证的是多轮、多来源和引用绑定的结构闭环。自动闸门只覆盖本例中的来源范围、物理行 quote、数字支持、实验边界和资料不足标记，不替代全文事实核验、权限控制、版本治理或生产评测。